In [ ]:
# Importing python libraries
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
data_of_interest = [
    # ---- DESENHO AMOSTRAL (OBRIGATÓRIO) ----
    "V0001",    # UF
    "V0024",    # Estrato
    "UPA_PNS",  # Unidade Primária de Amostragem (Faltava)
    "V0028",    # Peso do morador selecionado s/ calibração (Faltava)
    "V00281",   # Peso do morador selecionado c/ calibração (Faltava)
    "V0020",    # Ano de referência
    "V0026",    # Tipo de situação censitária (Urbano/Rural)

    # ---- DEMOGRÁFICAS E SOCIOECONÔMICAS ----
    "C006",     # Sexo (1 = Homem, 2 = Mulher)
    "C00701",   # Dia nascimento
    "C00702",   # Mês nascimento
    "C00703",   # Ano nascimento
    "C008",     # Idade na data de referência
    "C009",     # Cor ou raça
    "C01001",   # Cônjuge mora no domicílio
    "C014",     # Natureza da união
    "D00901",   # Escolaridade padronizada

    # ---- ANTROPOMETRIA / ESTILO DE VIDA (MORADOR SELECIONADO) ----
    "P00102",   # Sabe seu peso?
    "P00103",   # Peso informado (kg)
    "P00104",   # Peso final (kg)
    "P00402",   # Sabe sua altura?
    "P00403",   # Altura informada (cm)
    "P00404",   # Altura final (cm)
    "P005",     # Está grávida no momento?

    # ---- SAÚDE DA MULHER (CENÁRIO 1: MENOPAUSA) ----
    "R044",     # Histerectomia (Faltava)
    "R045",     # Ooforectomia (Faltava)
    "R046",     # Idade da última menstruação / menopausa (Faltava)

    # ---- PERCEPÇÃO DE SAÚDE, ANGINA E PHQ-9 (DEPRESSÃO) ----
    "N001",     # Autoavaliação de saúde geral
    "N00101",   # Autoavaliação de bem-estar biopsicossocial
    "N004", "N005", "N006", "N007", "N008", # Angina de peito (Rose)
    "N010", "N011", "N012", "N013", "N014", 
    "N015", "N016", "N017", "N018"          # Rastreio PHQ-9
]

In [3]:
base_dir = Path("../../dados_pns_2019").resolve()
txt_path = base_dir / "PNS_2019.txt"
doc_dir = base_dir / "documentacao"

In [4]:
# Localiza dinamicamente o arquivo Excel do dicionário
dict_files = list(doc_dir.glob("*.xls*"))
if not dict_files:
    raise FileNotFoundError(f"Dicionário não encontrado em {doc_dir}")
dict_path = dict_files[0]
print(f"Usando dicionário: {dict_path.name}")

Usando dicionário: dicionario_PNS_microdados_2019.xls


In [9]:
excel_file = pd.ExcelFile(dict_path)
print(f"Abas encontradas no Excel: {excel_file.sheet_names}")

# 1. Seleciona a aba que contém os microdados/dicionário
target_sheet = excel_file.sheet_names[0]
for sheet in excel_file.sheet_names:
    if any(k in sheet.lower() for k in ["dicion", "micro", "pns", "var"]):
        target_sheet = sheet
        break
print(f"Lendo aba: '{target_sheet}'")

# 2. Lê as primeiras 25 linhas brutas para localizar a linha do cabeçalho
df_raw = pd.read_excel(dict_path, sheet_name=target_sheet, header=None, nrows=25)

header_idx = None
for idx, row in df_raw.iterrows():
    row_text = " ".join(row.dropna().astype(str).str.lower())
    # O cabeçalho real sempre contém 'posi' e ('tamanho' ou 'tam')
    if "posi" in row_text and ("tamanho" in row_text or "tam" in row_text):
        header_idx = idx
        break

if header_idx is None:
    print("\n--- Primeiras 10 linhas da aba para diagnóstico ---")
    print(df_raw.head(10))
    raise ValueError("Não foi possível encontrar a linha de cabeçalho com 'Posição' e 'Tamanho'.")

print(f"Cabeçalho identificado na linha: {header_idx + 1}")

# 3. Carrega o dicionário a partir da linha de cabeçalho correta
df_dict = pd.read_excel(dict_path, sheet_name=target_sheet, skiprows=header_idx)
df_dict.columns = [str(c).strip() for c in df_dict.columns]

# Identifica as colunas relevantes independentemente de maiúsculas/acentos
col_pos = [c for c in df_dict.columns if "posi" in c.lower()][0]
col_tam = [c for c in df_dict.columns if "tamanho" in c.lower() or "tam" in c.lower()][0]
col_var = [c for c in df_dict.columns if any(k in c.lower() for k in ["vari", "código", "codigo"])][0]

print(f"Colunas mapeadas no Excel -> Posição: '{col_pos}' | Tamanho: '{col_tam}' | Variável: '{col_var}'")

Abas encontradas no Excel: ['dicionário pns']
Lendo aba: 'dicionário pns'
Cabeçalho identificado na linha: 2
Colunas mapeadas no Excel -> Posição: 'Posição inicial' | Tamanho: 'Tamanho' | Variável: 'Código
da
variável'


In [10]:
# 4. Limpa e converte os tipos
df_dict[col_var] = df_dict[col_var].astype(str).str.strip().str.upper()
df_dict = df_dict.dropna(subset=[col_pos, col_tam])

# Converte posição e tamanho para inteiros válidos
df_dict[col_pos] = pd.to_numeric(df_dict[col_pos], errors="coerce")
df_dict[col_tam] = pd.to_numeric(df_dict[col_tam], errors="coerce")
df_dict = df_dict.dropna(subset=[col_pos, col_tam])

In [11]:
# 5. Filtra pelas variáveis de interesse
vars_desejadas = [v.strip().upper() for v in data_of_interest]
df_layout = df_dict[df_dict[col_var].isin(vars_desejadas)].copy()

In [12]:
# Cria as fatias 0-based para o Python (start, end)
slices = []
col_names = []
for _, row in df_layout.iterrows():
    var_name = str(row[col_var]).strip()
    start_pos = int(row[col_pos]) - 1
    length = int(row[col_tam])
    end_pos = start_pos + length
    
    slices.append((var_name, start_pos, end_pos))
    col_names.append(var_name)

print(f"Variáveis mapeadas com sucesso: {len(slices)} de {len(data_of_interest)}")

Variáveis mapeadas com sucesso: 38 de 42


In [13]:
# Mostra quais variáveis da sua lista não foram encontradas (se houver)
nao_encontradas = set(vars_desejadas) - set(col_names)
if nao_encontradas:
    print(f"Variáveis não localizadas no dicionário: {nao_encontradas}")

Variáveis não localizadas no dicionário: {'R045', 'N007', 'R044', 'R046'}


In [15]:
print("Processando arquivo de dados...")
rows = []
with open(txt_path, "r", encoding="latin1") as f:
    for line in f:
        # Extrai somente os caracteres de cada variável e remove espaços excedentes
        row_data = [line[start:end].strip() for _, start, end in slices]
        rows.append(row_data)

# 5. Criação do DataFrame limpo
df = pd.DataFrame(rows, columns=[name for name, _, _ in slices])
df.replace("", np.nan, inplace=True)

print(f"Concluído: {df.shape[0]:,} linhas x {df.shape[1]} colunas carregadas.")

Processando arquivo de dados...
Concluído: 293,726 linhas x 38 colunas carregadas.
